Damped Newton method for chi=30

In [186]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [175]:
gilt_eps = 2e-5 #6e-6
chi = 34
trunc_shape = [14 14; 14 14; 14 14; 14 14]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
trunc_shape = [14 16; 14 16; 14 16; 14 16] 
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => [chi], #collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 2,
	"rotate" => true,
    "degeneracy_eps" => 1e-16
)
Jratio = 1.0

relT=1.0
rg_steps = 10
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

[ Info: Found the record with len=10
┌ Warning: construct_linear_system: unable to find 29 independent rows, will fix only 5 dofs
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:338
┌ Warning: construct_linear_system: unable to find 30 independent rows, will fix only 29 dofs
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:338
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 3.578955199556428e-5 and became -3.3927111004865723e-9. Index CartesianIndex(1, 1, 14, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 2.0816277913649415e-5 and became -5.224790372897466e-8. Index CartesianIndex(1, 18, 34, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -8.354797467237773e-6 and became -6.49168682467281e

In [170]:
[chi]

1-element Vector{Int64}:
 34

In [176]:
for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

1 [1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
3 [8 9; 8 8; 8 9; 8 8][0 1; 0 1; 0 1; 0 1]
4 [17 17; 17 17; 17 17; 17 17][0 1; 0 1; 0 1; 0 1]
5 [17 17; 17 17; 17 17; 17 17][0 1; 0 1; 0 1; 0 1]
6 [17 17; 17 17; 17 17; 17 17][0 1; 0 1; 0 1; 0 1]
7 [17 17; 17 17; 17 17; 17 17][0 1; 0 1; 0 1; 0 1]
8 [17 17; 17 17; 17 17; 17 17][0 1; 0 1; 0 1; 0 1]
9 [17 17; 17 17; 17 17; 17 17][0 1; 0 1; 0 1; 0 1]
10 [17 17; 17 17; 17 17; 17 17][0 1; 0 1; 0 1; 0 1]
11 [17 17; 17 17; 17 17; 17 17][0 1; 0 1; 0 1; 0 1]


In [177]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

In [178]:
A[1] = truncate_blocks(traj[4], trunc_shape);

In [166]:
A[1].shape

4×2 Matrix{Int64}:
 14  16
 14  16
 14  16
 14  16

In [167]:
A[1].qhape

4×2 Matrix{Int64}:
 0  1
 0  1
 0  1
 0  1

In [187]:
for i in 1:30
    println("i=",i)
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    e0, RAshape = fp_error_with_shape(A[i],accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RAshape)
    flush(stdout)
    if A[i].shape != RAshape
        throw(ErrorException("shapes unequal"))
    end
    deltaA[i] = newton_correction(A[i], 10, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 1.0
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]
        Anew, accepted_elements_new = fix_discrete_gauge(Anew; tol = 1e-7);
        enew, RAnewshape = fp_error_with_shape(Anew, accepted_elements_new, gilt_pars; trunc_shape = trunc_shape)
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnewshape)
        if enew < e0 && Anew.shape == RAnewshape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=1
gilt, A0 shape: [14 16; 14 16; 14 16; 14 16]
gilt, A0 qhape: [0 1; 0 1; 0 1; 0 1]
2024-12-29 09:25:18 I: Lap                                 1
2024-12-29 09:25:18 I: Gilt initiated at leg               S
2024-12-29 09:25:18 I: Leg status:                         False
2024-12-29 09:25:18 I: Gilt info,                          Error = 6.239e-05, shape = (np.int64(30), np.int64(30), 6, np.int64(30)) [(np.int64(30), np.int64(30), np.int64(30), np.int64(30))]
2024-12-29 09:25:18 I: Gilt initiated at leg               N
2024-12-29 09:25:18 I: Leg status:                         False
2024-12-29 09:25:18 I: Gilt info,                          Error = 1.270e-04, shape = (6, np.int64(30), 6, np.int64(30)) [(np.int64(30), np.int64(30), np.int64(30), np.int64(30))]
2024-12-29 09:25:18 I: Gilt initiated at leg               E
2024-12-29 09:25:18 I: Leg status:                         False
2024-12-29 09:25:18 I: Gilt info,                          Error = 1.901e-04, shape = (6, np.int64(30), 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.25692250109352366 and became -8.770082928434218e-8. Index CartesianIndex(15, 1, 1, 15) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.10335432588323676 and became 8.327416412343087e-8. Index CartesianIndex(16, 1, 1, 16) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.09468715077958445 and became -3.627622039221054e-8. Index CartesianIndex(1, 1, 15, 16) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.09057056857732641 and became -2.7769342393487104e-9. Index CartesianIndex(15, 1, 15, 2) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_e

LoadError: shapes unequal